<a href="https://colab.research.google.com/github/ayeung009/APS360-Project/blob/main/Final_Test_Data_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final Testing Dataset Generation

This notebook contains the work done to generate data for the final testing of the model.

Evaluate model on new data (10 points): Describe the efforts taken to ensure the results
are a good representation of the model’s performance on new data. Can you evaluate model
on new data? This will depend greatly on the problem being solved.

In [3]:
!git clone https://github.com/ayeung009/APS360-Project.git
%cd /content/APS360-Project
!pip install schemdraw pandas tqdm --quiet

Cloning into 'APS360-Project'...
remote: Enumerating objects: 1461, done.
remote: Counting objects: 100% (1461/1461), done.
remote: Compressing objects: 100% (1436/1436), done.
remote: Total 1461 (delta 79), reused 1308 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (1461/1461), 27.51 MiB | 16.79 MiB/s, done.
Resolving deltas: 100% (79/79), done.
/content/APS360-Project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.5/152.5 kB 4.4 MB/s eta 0:00:00


In [15]:
import os
import random
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from schemdraw.parsing import logicparse
from collections import Counter

NUM_SAMPLES = 300


PROJECT_ROOT = Path("/content/APS360-Project")
EXISTING_LABELS_CSV = PROJECT_ROOT / "data" / "final_merged_dataset" / "final_labels.csv"


OUTPUT_DIR = PROJECT_ROOT / "data" / "new_test_data"
IMAGE_DIR = OUTPUT_DIR / "images"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

NEW_SEED = 731
random.seed(NEW_SEED)

VARIABLES = ["A", "B", "C", "D", "E"]
BINARY_OPS = ["and", "or", "xor", "nand", "nor", "xnor"]


def negate(sub_expr):
    s = sub_expr.strip()
    if s.startswith("(not ") and s.endswith(")"):
        return s[5:-1]
    return f"(not {sub_expr})"


def generate_expression(variables):
    expressions = variables.copy()
    random.shuffle(expressions)

    if len(expressions) == 1:
        return negate(expressions[0])

    while len(expressions) > 1:
        index1 = random.randrange(len(expressions))
        a = expressions.pop(index1)
        index2 = random.randrange(len(expressions))
        b = expressions.pop(index2)

        if random.random() < 0.25:
            a = negate(a)
        if random.random() < 0.25:
            b = negate(b)

        op = random.choice(BINARY_OPS)
        combined = f"({a} {op} {b})"

        if random.random() < 0.15:
            combined = negate(combined)

        expressions.append(combined)

    return expressions[0]


existing_df = pd.read_csv(EXISTING_LABELS_CSV)
seen = set(existing_df["expression"].tolist())
print(f"Loaded {len(seen)} unique existing expressions to exclude.")

records = []
counter = 0
skipped_duplicates = 0

print(f"Generating {NUM_SAMPLES} new, never-before-seen test circuits...")
with tqdm(total=NUM_SAMPLES) as pbar:
    while counter < NUM_SAMPLES:
        n_inputs = random.randint(1, 5)
        vars_used = random.sample(VARIABLES, n_inputs)
        expression = generate_expression(vars_used)

        if expression in seen:
            skipped_duplicates += 1
            continue

        seen.add(expression)

        filename = f"{counter:05d}.png"
        filepath = IMAGE_DIR / filename
        drawing = logicparse(expression)
        drawing.save(str(filepath), dpi=300)

        records.append({
            "image_path": str(Path("data") / "new_test_data" / "images" / filename),
            "expression": expression,
            "source": "final_test_data",
            "n_inputs": n_inputs,
        })

        counter += 1
        pbar.update(1)

df = pd.DataFrame(records)
labels_path = OUTPUT_DIR / "labels.csv"
df.to_csv(labels_path, index=False)

print(f"\nGenerated {len(df)} new test images.")
print(f"Skipped {skipped_duplicates} generated expressions that already existed in the training/val/test data.")
print(f"Saved to: {labels_path}")
print()
print(df["n_inputs"].value_counts().sort_index())

Loaded 1013 unique existing expressions to exclude.
Generating 300 new, never-before-seen test circuits...


100%|██████████| 300/300 [00:41<00:00,  7.21it/s]


Generated 300 new test images.
Skipped 114 generated expressions that already existed in the training/val/test data.
Saved to: /content/APS360-Project/data/new_test_data/labels.csv

n_inputs
2     45
3     74
4    104
5     77
Name: count, dtype: int64


# Important Cross-Check
Ensure that all data is never seen before. That is, all data in the new test set does not already exist in the original training data.

In [18]:
import pandas as pd
new_df = pd.read_csv("data/new_test_data/labels.csv")
existing_df = pd.read_csv("data/final_merged_dataset/final_labels.csv")

print("Internal duplicates:", new_df["expression"].duplicated().sum())
print("Overlap with existing:", len(set(new_df["expression"]) & set(existing_df["expression"])))

Internal duplicates: 0
Overlap with existing: 0


In [19]:
#Download dataset locally
!zip -r new_test_data_images.zip /content/APS360-Project/data/new_test_data/images

  adding: content/APS360-Project/data/new_test_data/images/ (stored 0%)
  adding: content/APS360-Project/data/new_test_data/images/00259.png (deflated 3%)
  adding: content/APS360-Project/data/new_test_data/images/00004.png (deflated 3%)
  adding: content/APS360-Project/data/new_test_data/images/00037.png (deflated 4%)
  adding: content/APS360-Project/data/new_test_data/images/00189.png (deflated 6%)
  adding: content/APS360-Project/data/new_test_data/images/00204.png (deflated 6%)
  adding: content/APS360-Project/data/new_test_data/images/00003.png (deflated 5%)
  adding: content/APS360-Project/data/new_test_data/images/00116.png (deflated 4%)
  adding: content/APS360-Project/data/new_test_data/images/00246.png (deflated 3%)
  adding: content/APS360-Project/data/new_test_data/images/00134.png (deflated 1%)
  adding: content/APS360-Project/data/new_test_data/images/00137.png (deflated 2%)
  adding: content/APS360-Project/data/new_test_data/images/00019.png (deflated 3%)
  adding: conte

In [20]:
from google.colab import files
files.download("new_test_data_images.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>